# 05 - EDA inicial

**Objetivo.** Realizar una primera exploración de la base consolidada limpia, con foco en cobertura, precios, superficie, distribución territorial y atributos de publicación. Este notebook no usa fuentes externas todavía.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Carga de base consolidada más avanzada disponible

In [ ]:
candidates = [
    PROCESSED_DIR / 'df_publicaciones_consolidadas_con_outliers_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_con_flags_nulos_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_v1.csv',
]
for path in candidates:
    if path.exists():
        CONSOLIDATED_PATH = path
        break
else:
    raise FileNotFoundError('No existe base consolidada. Ejecutar notebooks 02-04.')

df_eda = pd.read_csv(CONSOLIDATED_PATH, low_memory=False)
print('Base usada:', CONSOLIDATED_PATH.relative_to(REPO_ROOT))
print(df_eda.shape)
display(df_eda.head())

## Cobertura por fuente y operación

In [ ]:
display(pd.crosstab(df_eda['fuente_norm'], df_eda['tipo_operacion_norm'], dropna=False))

## Cobertura territorial

In [ ]:
barrios = df_eda['barrio_norm'].value_counts(dropna=False).head(30)
display(barrios.to_frame('publicaciones'))

try:
    import matplotlib.pyplot as plt
    barrios.sort_values().plot(kind='barh', figsize=(10, 8), title='Top 30 barrios por cantidad de publicaciones')
    plt.xlabel('Publicaciones')
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(exc)

## Calidad remanente de campos críticos

In [ ]:
campos = ['precio', 'moneda_norm', 'barrio_norm', 'superficie_ref_m2', 'precio_m2_ref']
resumen_calidad = df_eda.groupby(['fuente_norm', 'tipo_operacion_norm'])[campos].apply(lambda x: x.isna().mean().mul(100).round(1))
display(resumen_calidad)

## Distribución inicial de precios por operación y moneda

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    import matplotlib.pyplot as plt
    HAS_SEABORN = False

plot_df = df_eda[df_eda['precio'].notna() & df_eda['moneda_norm'].notna()].copy()
plot_df = plot_df[plot_df['precio'] > 0]

partes_filtradas = []
for _, grupo in plot_df.groupby(['tipo_operacion_norm', 'moneda_norm']):
    q_low = grupo['precio'].quantile(0.01)
    q_high = grupo['precio'].quantile(0.99)
    partes_filtradas.append(grupo[grupo['precio'].between(q_low, q_high)])

plot_df = pd.concat(partes_filtradas, ignore_index=True) if partes_filtradas else pd.DataFrame()

if HAS_SEABORN and not plot_df.empty:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=plot_df, x='tipo_operacion_norm', y='precio', hue='moneda_norm')
    plt.yscale('log')
    plt.title('Distribucion de precios por operacion y moneda (p01-p99, escala log)')
    plt.tight_layout()
    plt.show()

## Precio por m2 y superficie

In [ ]:
metricas = df_eda.groupby(['fuente_norm', 'tipo_operacion_norm', 'moneda_norm'])[['precio_m2_ref', 'superficie_ref_m2']].agg(['count', 'median', 'mean']).round(2)
display(metricas)

## Atributos y amenities comparables

In [ ]:
amenity_cols = [c for c in ['balcon', 'balcon_o_patio', 'terraza', 'pileta', 'parrilla', 'cochera', 'cochera_mencionada', 'amenities_mencionadas'] if c in df_eda.columns]
if amenity_cols:
    resumen_amenities = df_eda.groupby(['fuente_norm', 'tipo_operacion_norm'])[amenity_cols].mean(numeric_only=True).mul(100).round(1)
    display(resumen_amenities)

## Cierre EDA inicial

Completar con interpretación luego de revisar las salidas:

- fuentes/operaciones con mejor cobertura;
- barrios con volumen suficiente;
- campos críticos con limitaciones;
- primeras señales descriptivas para hipótesis;
- análisis que pasan a fuentes externas.